# @tool 装饰器——定义工具

工具（Tool）是 Agent 与外部世界交互的桥梁。通过 `@tool` 装饰器，你可以将任何 Python 函数快速转换为 Agent 可调用的工具。

In [1]:
from dotenv import load_dotenv
load_dotenv()

print("环境变量已加载")

环境变量已加载


## @tool 基本语法

在函数上加上 `@tool` 装饰器，函数就变成了一个工具。函数文档字符串自动成为工具的描述。

In [1]:
from langchain.tools import tool

@tool
def hello_tool(name: str) -> str:
    """向指定的人打招呼。

    Args:
        name: 要打招呼的人的名字
    """
    return f"你好，{name}！欢迎来到中国。"

# 工具也是普通的 Python 函数，可以直接调用
result = hello_tool.invoke({"name": "小明"})
print(result)

print(f"\n工具名称: {hello_tool.name}")
print(f"工具描述: {hello_tool.description[:60]}...")

你好，小明！欢迎来到中国。

工具名称: hello_tool
工具描述: 向指定的人打招呼。

    Args:
        name: 要打招呼的人的名字...


> 文档字符串会自动成为工具的描述，Agent 依赖它判断工具用途。写得越清晰，Agent 使用越准确。

## 使用不同参数类型

`@tool` 支持多种参数类型，包括 int、float、bool 和枚举值。

In [3]:
from langchain.tools import tool
from typing import Literal

@tool
def search_courses(
    keyword: str,
    level: Literal["入门", "进阶", "高级"],
    max_results: int = 5,
    free_only: bool = True,
) -> str:
    """在菜鸟教程 RUNOOB 中搜索课程。

    Args:
        keyword: 搜索关键词
        level: 课程难度级别
        max_results: 最多返回多少条结果
        free_only: 是否只显示免费课程
    """
    courses = {
        ("python", "入门"): "Python3 基础教程（免费）",
        ("python", "进阶"): "Python 数据分析实战（免费）",
        ("python", "高级"): "Python 机器学习进阶（会员）",
        ("html", "入门"): "HTML 基础教程（免费）",
    }
    key = (keyword.lower(), level)
    course = courses.get(key, f"未找到 {level} 级别的 {keyword} 课程")
    if free_only and "会员" in course:
        return f"搜索结果：{course} —— 该课程需要会员，已为你过滤"
    return f"搜索结果：{course}"

print(search_courses.invoke({"keyword": "python", "level": "入门"}))
print(search_courses.invoke({"keyword": "python", "level": "高级", "free_only": False}))

搜索结果：Python3 基础教程（免费）
搜索结果：Python 机器学习进阶（会员）


## 注册工具到 Agent

将工具传给 `create_agent()` 的 `tools` 参数即可。

In [5]:
from langchain.tools import tool
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage

@tool
def search_courses_web(keyword: str) -> str:
    """在菜鸟教程 RUNOOB 中搜索课程。传入关键词返回相关课程列表。

    Args:
        keyword: 搜索关键词，如 python、html、java
    """
    courses = {
        "python": "Python3 基础教程、Python 数据分析、Python 爬虫入门",
        "html": "HTML 基础教程、HTML5 新特性、HTML 表单实战",
        "java": "Java 基础教程、Java 面向对象、Java Spring 框架",
    }
    return courses.get(keyword.lower(), f"未找到与 {keyword} 相关的课程")

@tool
def get_course_detail(course_name: str) -> str:
    """获取指定课程的详细信息，包括章节数和学习时长。

    Args:
        course_name: 课程名称
    """
    details = {
        "python3 基础教程": "共 30 章，预计学习时长 20 小时，适合零基础入门",
        "html 基础教程": "共 25 章，预计学习时长 15 小时，适合零基础入门",
    }
    return details.get(
        course_name.lower(),
        f"{course_name} 详情：适合初学者，内容丰富，附带实战案例"
    )

model = init_chat_model("deepseek:deepseek-v4-flash", temperature=0)
agent = create_agent(
    model=model,
    tools=[search_courses_web, get_course_detail],
    system_prompt="你是菜鸟教程 RUNOOB 的学习顾问，帮助用户找到合适的课程。",
)

result = agent.invoke({"messages": [HumanMessage(content="我想学 Python，有什么课程推荐？")]})
print(f"回答: {result['messages'][-1].content}")

回答: 太好了！我为你整理了菜鸟教程上的 **Python 相关课程**，以下是推荐👇

---

### 🏆 推荐课程一览

| 课程名称 | 特点 | 适合人群 |
|---------|------|---------|
| **① Python3 基础教程** | 📘 共 **30 章**，学习时长约 **20 小时**，零基础友好 ✅ | **新手入门首选！** |
| **② Python 数据分析** | 📊 内容丰富，附带实战案例 | 有基础想进阶数据分析 |
| **③ Python 爬虫入门** | 🕷️ 内容丰富，附带实战案例 | 有基础想学爬虫技术 |

---

### 🎯 我的强烈建议

如果你是 **零基础** 想学 Python，我建议你从 **① Python3 基础教程** 开始：
- ✅ 专门为零基础设计
- ✅ 30 个章节系统学习
- ✅ 预计 20 小时即可入门
- ✅ 打好基础后，再去学数据分析或爬虫会事半功倍！

你是 **完全零基础** 还是已经有一些编程经验了呢？我可以根据你的情况给你更精准的建议～😊


## 多个工具协作

Agent 自动判断何时使用哪个工具。

In [6]:
from langchain.tools import tool
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage

@tool
def get_stock_price(symbol: str) -> str:
    """查询股票当前价格。

    Args:
        symbol: 股票代码，如 AAPL、GOOGL、TSLA
    """
    prices = {"AAPL": 185.50, "GOOGL": 142.30, "TSLA": 245.80}
    price = prices.get(symbol.upper())
    return f"{symbol.upper()} 当前价格：${price}" if price else f"未找到 {symbol}"

@tool
def convert_currency(amount: float, from_currency: str, to_currency: str) -> str:
    """货币换算。

    Args:
        amount: 金额
        from_currency: 原货币代码
        to_currency: 目标货币代码
    """
    rates = {("USD", "CNY"): 7.25, ("CNY", "USD"): 0.138}
    rate = rates.get((from_currency.upper(), to_currency.upper()))
    result = round(amount * rate, 2)
    return f"{amount} {from_currency} = {result} {to_currency}"

model = init_chat_model("deepseek:deepseek-v4-flash", temperature=0)
agent = create_agent(
    model=model,
    tools=[get_stock_price, convert_currency],
    system_prompt="你是一个金融助手。",
)

result = agent.invoke({"messages": [HumanMessage(content="苹果股价多少？换算成人民币？")]})
for msg in result["messages"]:
    if msg.type == "tool":
        print(f"[工具 {msg.name}] {msg.content}")
    elif msg.type == "ai" and msg.content:
        print(f"\n回答: {msg.content}")

print("金融助手已创建")


回答: 好的！我来查一下苹果的当前股价，同时获取一下美元兑人民币的汇率。
[工具 get_stock_price] AAPL 当前价格：$185.5

回答: 苹果当前股价为 **$185.5 美元**。现在我把这个金额换算成人民币。
[工具 convert_currency] 185.5 USD = 1344.88 CNY

回答: 以下是查询结果汇总：

### 🍎 苹果（AAPL）股价信息

| 项目 | 金额 |
|------|------|
| **美股价格** | **$185.50 美元** |
| **换算成人民币** | **¥1,344.88 元** |

> *汇率按实时换算，仅供参考。实际汇率可能因市场波动略有变化。*

苹果股价目前为 **185.5 美元/股**，折合人民币约 **1,344.88 元/股**。如果您想进一步了解其他信息，比如历史走势或买卖建议，欢迎随时问我！😊
金融助手已创建


## 工具的可选参数与默认值

为参数设置默认值，让工具更灵活。

In [7]:
from langchain.tools import tool

@tool
def recommend_tutorial(
    language: str,
    level: str = "入门",
    count: int = 3,
) -> str:
    """根据编程语言和难度推荐菜鸟教程的课程。

    Args:
        language: 编程语言
        level: 难度级别
        count: 推荐课程数量
    """
    tutorials = {
        "python": ["Python3 基础", "Python 面向对象", "Python 爬虫", "Python 数据分析"],
        "java": ["Java 基础", "Java 面向对象", "Java 集合框架", "Java 多线程"],
    }
    all_tutorials = tutorials.get(language.lower(), [f"{language} 基础教程"])
    selected = all_tutorials[:count]
    return f"推荐 {language} {level} 级别课程：{'、'.join(selected)}"

print(recommend_tutorial.invoke({"language": "Python"}))
print(recommend_tutorial.invoke({"language": "Java", "level": "进阶", "count": 2}))

推荐 Python 入门 级别课程：Python3 基础、Python 面向对象、Python 爬虫
推荐 Java 进阶 级别课程：Java 基础、Java 面向对象


> 默认值让 Agent 不用每次都指定所有参数。但关键参数不要设默认值，避免 Agent 遗漏。

## 工具的 args_schema——自定义参数校验

使用 Pydantic 模型实现精细参数控制。

In [9]:
from pydantic import BaseModel, Field
from langchain.tools import tool

class CourseSearchInput(BaseModel):
    """搜索课程参数"""
    keyword: str = Field(description="搜索关键词", min_length=1, max_length=50)
    category: str = Field(
        default="all",
        description="课程类别",
        pattern=r"^(all|frontend|backend|data)$",
    )
    page: int = Field(default=1, description="页码", ge=1, le=100)

@tool(args_schema=CourseSearchInput)
def search_course_validated(keyword: str, category: str = "all", page: int = 1) -> str:
    """在菜鸟教程 RUNOOB 中搜索课程"""
    return f"搜索 '{keyword}' (分类: {category}, 第 {page} 页)：共找到 15 条结果"

print(search_course_validated.invoke({"keyword": "Python", "category": "backend"}))

try:
    search_course_validated.invoke({"keyword": "Python", "category": "invalid"})
except Exception as e:
    print(f"参数校验失败: {e}")

搜索 'Python' (分类: backend, 第 1 页)：共找到 15 条结果
参数校验失败: 1 validation error for CourseSearchInput
category
  String should match pattern '^(all|frontend|backend|data)$' [type=string_pattern_mismatch, input_value='invalid', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/string_pattern_mismatch


## 工具定义方式对比

| 方式 | 代码量 | 适用场景 |
| --- | --- | --- |
| @tool 装饰器 | 最少 | 简单到中等复杂度的工具 |
| @tool + args_schema | 中等 | 需要精细参数校验 |
| Pydantic 类作为工具 | 较多 | 复杂业务逻辑 |
| 字典格式 | 最少（不推荐） | 描述远程/内置工具 |

**术语：**

- **@tool**：LangChain 装饰器，将函数转为工具
- **args_schema**：Pydantic 模型，定义工具参数校验规则
- **tool.invoke()**：直接调用工具（传入 dict 参数）
- **工具描述**：函数的文档字符串，Agent 判断何时使用的依据